# 04 · Evaluate exploratory synthetic segmentation

KMeans is an analytical exercise on invented attitudes, not discovery of natural customer segments. **Evidence status: SYNTHETIC.** Illustrative results on independently generated synthetic data; not findings from the ADK consumer survey.

This notebook generates and validates its own fixture. The pipeline imputes missing attitudes, standardizes them, and chooses PCA dimensions using retained variance. IDs and behavior fields are excluded from clustering. See [analytical limitations](../docs/ANALYTICAL_LIMITATIONS.md).

In [ ]:
import pandas as pd
from IPython.display import display

from retail_case.schema import SYNTHETIC_NOTICE
from retail_case.synthetic import generate_data

df = generate_data(n=720, seed=2026, missing_rate=0.03)


def show_synthetic_table(title, table):
    print(f"SYNTHETIC — {title}")
    print(SYNTHETIC_NOTICE)
    display(table)

In [ ]:
from retail_case.clustering import analyze_data
from retail_case.preprocess import prepare_features

features, preprocessing = prepare_features(df)
show_synthetic_table(
    "Preprocessing metadata and PCA rationale", pd.DataFrame({"metadata": [preprocessing]})
)
print(f"SYNTHETIC transformed matrix shape: {features.shape}")

result = analyze_data(df)
candidates = pd.DataFrame(result["candidates"])
show_synthetic_table(
    "Candidate k comparison: fit, size balance, and initialization agreement", candidates
)
show_synthetic_table(
    "Sensitivity to representation and preprocessing", pd.DataFrame(result["sensitivity"])
)
print(
    f"SYNTHETIC technical illustration k: {result['exploratory_k']}; "
    "selected for comparison, not a validated audience decision."
)
for warning in result["warnings"]:
    print(f"SYNTHETIC analytical caveat: {warning}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].plot(candidates["k"], candidates["silhouette"], marker="o")
axes[0].set(title="SYNTHETIC silhouette by k", xlabel="Candidate k", ylabel="Silhouette")
axes[1].plot(
    candidates["k"],
    candidates["initialization_ari_mean"],
    marker="o",
    label="Mean initialization ARI",
)
axes[1].plot(
    candidates["k"],
    candidates["initialization_ari_min"],
    marker="s",
    label="Minimum initialization ARI",
)
axes[1].set(
    title="SYNTHETIC initialization agreement", xlabel="Candidate k", ylabel="Adjusted Rand index"
)
axes[1].legend(fontsize="small")
for axis in axes:
    axis.set_xticks(candidates["k"])
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## What model comparisons can establish

**Observation:** silhouette, partition sizes, repeated-initialization ARI, and representation sensitivity describe behavior of this fitted model on this fixture. **Interpretation:** higher ARI indicates similar assignments under the tested initializations. It does not validate real segments, attitude measurement, prediction, or business usefulness.

**Recommendation:** inspect multiple candidates and the unsegmented/descriptive baseline before using any partition in a pilot. The technical reference maximizes observed silhouette, with ties preferring smaller k; this is only a reproducible illustration rule. No favorable metric is hard-coded. A no-cluster outcome remains meaningful even when an algorithm always returns labels. [05 · Targeting](05_targeting.ipynb) compares profiles without declaring an objectively best audience.